<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/split_main_prod_to_a_half_sized_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Setup paths
base_dir = '/content/drive/MyDrive/world-inflation/data/reddit/production/'
input_file = '2-main-prod-1039.csv'
input_path = os.path.join(base_dir, input_file)

# Load original dataset
df = pd.read_csv(input_path)

print(f"==============================\nOriginal Data\n==============================")
print(f"Size: {len(df)}")
counts = df['inflation'].value_counts().sort_index()
ratios = df['inflation'].value_counts(normalize=True).sort_index()
for cls in counts.index:
    print(f"Class {cls}: {counts[cls]} ({ratios[cls]:.1%})")

# Target sizes descending
target_sizes = [520, 260, 130, 65]
current_df = df.copy()

for size in target_sizes:
    print(f"\n==============================\nCreating subset of size: {size}\n==============================")

    # Stratified sampling to maintain ORIGINAL distribution within the nested subset
    # This ensures Class 0 is ~30%, Class 1 is ~38%, Class 2 is ~31% at every step
    train_df, _ = train_test_split(
        current_df,
        train_size=size,
        stratify=current_df['inflation'],
        random_state=42
    )

    # Calculate and display stats
    counts = train_df['inflation'].value_counts().sort_index()
    ratios = train_df['inflation'].value_counts(normalize=True).sort_index()

    print(f"Dataset Size: {len(train_df)}")
    for cls in counts.index:
        print(f"Class {cls}: {counts[cls]} ({ratios[cls]:.1%})")

    # Save file
    output_filename = f"2-main-prod-{size}.csv"
    output_path = os.path.join(base_dir, output_filename)
    train_df.to_csv(output_path, index=False)

    # Update current_df to ensure strict nesting (Subset constraint)
    current_df = train_df

print("\nAll files created successfully with preserved class ratios.")